# 03 - Pointwise Intensity Operations

A pointwise transformation computes each output pixel from the corresponding input pixel:

\[
g(x,y)=T(f(x,y)),
\]

where \(f(x,y)\) is the input intensity, \(T\) is a scalar mapping, and \(g(x,y)\) is the output intensity. The experiments compare inversion, affine brightness/contrast control, gamma correction, and lookup-table implementation.

In [ ]:
from pathlib import Path
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_course_root() -> Path:
    """Locate the cloned course repository without a machine-specific path."""
    candidates = []
    configured = os.environ.get("VISION_ROBOTICA_REPO")
    if configured:
        candidates.append(Path(configured).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "imagenes").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Course repository not found. Run the notebook from the cloned repository root "
        "or define VISION_ROBOTICA_REPO."
    )

ROOT = find_course_root()
COURSE_IMAGES = ROOT / "imagenes"
STUDENT_IMAGES = ROOT / "student_work" / "imagenes"
STUDENT_IMAGES.mkdir(parents=True, exist_ok=True)

def load_rgb(filename: str) -> np.ndarray:
    path = COURSE_IMAGES / filename
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Official course image not found: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print(f"Course root: {ROOT}")
print(f"Official images: {COURSE_IMAGES}")
print(f"Student images: {STUDENT_IMAGES}")

In [ ]:
image_rgb = load_rgb("barco.png")
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
L = 255.0

negative = 255 - gray

alpha, beta = 1.35, 20.0
affine = np.clip(alpha * gray.astype(np.float32) + beta, 0, L).astype(np.uint8)

gamma = 0.55
normalized = gray.astype(np.float32) / L
gamma_corrected = np.round(L * normalized**gamma).astype(np.uint8)

lookup = np.round(L * (np.arange(256, dtype=np.float32) / L)**gamma).astype(np.uint8)
gamma_lut = cv2.LUT(gray, lookup)
assert np.max(np.abs(gamma_corrected.astype(int) - gamma_lut.astype(int))) <= 1

results = {
    "Original": gray,
    "Negative": negative,
    f"Affine: alpha={alpha}, beta={beta}": affine,
    f"Gamma: gamma={gamma}": gamma_corrected,
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, (title, image) in zip(axes.ravel(), results.items()):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## Saturation and contrast measurements

The saturation ratio is

\[
\rho_{\mathrm{sat}}=\frac{N_0+N_{255}}{MN},
\]

where \(N_0\) and \(N_{255}\) count pixels clipped to the two limits and \(M\times N\) is the image size. Standard deviation is reported as a global contrast indicator; it does not describe spatial arrangement.

In [ ]:
def saturation_ratio(image: np.ndarray) -> float:
    return float(np.mean((image == 0) | (image == 255)))

print(f"{'Transformation':34s} {'Mean':>9s} {'Std':>9s} {'Saturation':>12s}")
for title, image in results.items():
    print(f"{title:34s} {image.mean():9.2f} {image.std():9.2f} {saturation_ratio(image):12.4f}")

fig, axis = plt.subplots(figsize=(9, 4))
for title, image in results.items():
    histogram = cv2.calcHist([image], [0], None, [256], [0, 256]).ravel()
    axis.plot(histogram, label=title, linewidth=1.3)
axis.set(xlabel="Intensity", ylabel="Pixel count", xlim=(0, 255))
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Robotic interpretation

- Gamma below one can reveal features in underexposed regions, but noise may become more visible.
- Positive brightness offsets can saturate reflective surfaces and remove inspection evidence.
- A useful operating point preserves the features required for localization, tracking, or inspection rather than maximizing visual contrast alone.

**Check:** vary \(\alpha\), \(\beta\), and \(\gamma\). Report one setting that improves dark detail without producing an unacceptable saturation ratio.